# Differential RV recovery - permissive gradient mask diagnostics

Copy of `rv_recovery_sim_line_by_line.ipynb` aimed at **mask diagnostics**,
not a frozen packaged-ccfopt assert.

1. Build a **permissive** gradient-centroid delta mask
   (`RELAXED_TEMPLATE_MASK_BUILD_KW` / `build_line_mask_from_template`) under
   `OUT` (does **not** overwrite packaged masks).
2. Run the full +/-150 km/s x 5 m/s bulk grid with line-by-line CCF on that
   full line list.
3. For each bulk, histogram per-line `(RV_B - RV_A) - 5 m/s` residuals and
   track which mask lines fail the **10 cm/s** gate.
4. Recommend a culled line list: reject lines that fail in **7/7** bulks;
   keep lines that fail in at most **2/7** (and are measurable at all 7).

By default this **reuses blaze L2s** from a prior E2E tree (same fiberflat /
WAVE / spectrum-Doppler products) and only re-runs stellar RV with the new
mask. Set `REBUILD_E2E = True` to densify/UTR/Stage0-2 from scratch (1-3 h).

Outputs under `OUT` (default
`/Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag`).


In [1]:
import os

import hispecdrp

print(f"hispecdrp {hispecdrp.__version__} @ {hispecdrp.__file__}")

# Diagnostics live here; KOA cache shared with sibling RV recovery runs.
OUT = "/Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag"
os.makedirs(OUT, exist_ok=True)

os.environ.setdefault(
    "KOA_CALIBRATION_CACHE",
    "/Users/ckilpatrick/hispec_rv_recovery_sim/cals/KOA_CALIBRATION_CACHE",
)
print("output:", OUT)
print("KOA_CALIBRATION_CACHE:", os.environ["KOA_CALIBRATION_CACHE"])


hispecdrp 0.0.post2+waveinject.local @ /Users/ckilpatrick/Library/CloudStorage/Dropbox/scripts/python/cursor/HISPEC_DRP/hispecdrp/__init__.py
output: /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag
KOA_CALIBRATION_CACHE: /Users/ckilpatrick/hispec_rv_recovery_sim/cals/KOA_CALIBRATION_CACHE


## 1. Sync / verify KOA caldb (BSPEC)

Same as the sibling recovery notebook. Keep offline after seeding.


In [2]:
from hispecdrp.calibrations import ensure_koa_calibrations

SPECTROGRAPH = "BSPEC"

# Offline-friendly: seed from other local caches if needed, do not hit KOA remote.
summary = ensure_koa_calibrations(
    SPECTROGRAPH,
    seed_sources=None,
    connect_remote=False,
    ignore_missing_cal_types=["lfc_wavesol"],
)
os.environ["HISPEC_KOA_CONNECT_REMOTE"] = "0"

seed = summary.get("seed") or {}
if seed.get("seeded"):
    print(
        f"seeded KOA cache from {seed['source']}  "
        f"({seed['n_fits_before']} -> {seed['n_fits_after']} FITS)"
    )
elif seed.get("n_fits_before", 0):
    print(f"KOA cache already populated ({seed['n_fits_before']} FITS)")
print(f"remote_mode: {summary.get('remote_mode')}")
if summary.get("remote_error"):
    print(f"remote_error: {summary['remote_error']}")
print("HISPEC_KOA_CONNECT_REMOTE:", os.environ.get("HISPEC_KOA_CONNECT_REMOTE"))
print(
    f"{summary['spectrograph']} caldb: {summary['n_records']} record(s); "
    f"downloaded {summary['n_downloaded']}/{summary['n_missing']} missing file(s)"
)
print("types:", summary["cal_types"])


[2026-08-19T12:36:48][hispecdrp.calibrations.koa_sync:353][INFO]: KOA cache already populated (16 FITS) at /Users/ckilpatrick/hispec_rv_recovery_sim/cals/KOA_CALIBRATION_CACHE; skip seed.
[2026-08-19T12:36:48][koa_middleware.store:186][INFO]: Cache initialized: cache_dir='/Users/ckilpatrick/hispec_rv_recovery_sim/cals/KOA_CALIBRATION_CACHE', data_dir='/Users/ckilpatrick/hispec_rv_recovery_sim/cals/KOA_CALIBRATION_CACHE/calibrations/hispec/', db='/Users/ckilpatrick/hispec_rv_recovery_sim/cals/KOA_CALIBRATION_CACHE/database/hispec_calibrations.db', table='hispec'
[2026-08-19T12:36:48][koa_middleware.store:468][WARNING]: Found 1 calibration file(s) recorded in the DB but missing from the local cache.
[2026-08-19T12:36:48][koa_middleware.store:468][WARNING]: Found 1 calibration file(s) recorded in the DB but missing from the local cache.
[2026-08-19T12:36:48][koa_middleware.store:468][WARNING]: Found 1 calibration file(s) recorded in the DB but missing from the local cache.
[2026-08-19T12:

## 2. Config + permissive gradient mask

Builds `OUT/<template>.relaxed.gradient.delta.mas` with
`RELAXED_TEMPLATE_MASK_BUILD_KW` (gradient centroids, depth ≥ 10%, **no**
deblend / telluric / saturated-core gates). Packaged masks are not touched.

`REBUILD_E2E=False` (default): reuse blaze `_spec.fits` from `L2_SOURCE_OUT`
(prior line-by-line E2E) and only re-run stellar RV with the new mask.


In [3]:
from pathlib import Path

import numpy as np

from hispecdrp.utils.line_mask_utils import (
    CCFOPT_SINGLE_LINE_STELLAR_RV_KW,
    RELAXED_TEMPLATE_MASK_BUILD_KW,
    MaskLines,
    build_line_mask_from_template,
)

FIBERS = [2]
INJECT_RV_VIA = "spectrum"
EXTRACTION_APERTURE = 8
TEMPLATE = "lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.h5"
TEMPLATE_PATH = str(
    Path(hispecdrp.__file__).resolve().parent / "data" / "templates" / TEMPLATE
)

POISSON_NOISE = False
CONVERT_TO_UINT16 = False
SEED = 0
DRV_M_S = 5.0
REQUIREMENT_M_S = 0.10  # 10 cm/s per-line residual gate for diagnostics
# Keep nominal bulks so L2_SOURCE_OUT directory labels match existing products.
# Off-grid inject tests live in rv_recovery_sim_line_by_line.ipynb.
BULK_RV_KM_S = [-150.0, -100.0, -50.0, 0.0, 50.0, 100.0, 150.0]

# Reuse prior E2E blaze L2s (science products); only stellar RV is re-run.
REBUILD_E2E = False
L2_SOURCE_OUT = "/Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line"

# Match observed RVCCF rows back onto the rest-frame mask line list.
MASK_MATCH_HALF_M_S = 5000.0
# Recommended keep: measurable at all bulks and fail count <= this.
MAX_FAILS_FOR_RECOMMENDED = 2

MASK_PATH = str(
    Path(OUT) / f"{Path(TEMPLATE).stem}.relaxed.gradient.delta.mas"
)


def bulk_run_label(bulk_km_s: float, which: str) -> str:
    sign = "m" if bulk_km_s < 0 else "p"
    return f"bulk_{sign}{abs(bulk_km_s):06.1f}kms_{which}".replace(".", "p")


def stellar_rv_primitives(
    ccf_input_velocity_m_s: float, *, save_preview: bool = False
) -> dict:
    kw = CCFOPT_SINGLE_LINE_STELLAR_RV_KW
    return {
        "algorithm": "ccf_line_mask",
        "mask_path": MASK_PATH,
        "ccf_input_velocity": float(ccf_input_velocity_m_s),
        "do_fibers": list(FIBERS),
        "window_line_by_line": True,
        "window_size_um": float(kw["window_size_um"]),
        "v_half_window": float(kw["v_half_window_m_s"]),
        "v_step_coarse": float(kw["v_step_m_s"]),
        "mask_auto_center_scout": bool(kw["mask_auto_center_scout"]),
        "mask_ccf_pixel_interp_order": int(kw["mask_ccf_pixel_interp_order"]),
        "ccf_min_good_pixels": int(kw["ccf_min_good_pixels"]),
        "ccf_zb": 0.0,
        "ccf_zb_from_barycorr": False,
        "save_preview": bool(save_preview),
        "preview_write_png": False,
        "show_preview": False,
    }


assert Path(TEMPLATE_PATH).is_file(), TEMPLATE_PATH
# Clip to BSPEC science coverage so line-by-line CCF stays tractable (~1.7k lines
# vs ~9k over the full 0.8-3 um template).
_mask_path, _entries = build_line_mask_from_template(
    TEMPLATE_PATH,
    output_path=MASK_PATH,
    wl_min_um=0.97,
    wl_max_um=1.34,
    **RELAXED_TEMPLATE_MASK_BUILD_KW,
)
MASK_LINES = MaskLines.load(MASK_PATH)
MASK_LAMBDA_UM = np.asarray(MASK_LINES.line_lambda_um, dtype=np.float64)
MASK_WEIGHT = np.asarray(MASK_LINES.weight, dtype=np.float64)
print(f"wrote permissive gradient mask: {MASK_PATH}")
print(f"n_lines = {len(MASK_LAMBDA_UM)}  (RELAXED_TEMPLATE_MASK_BUILD_KW)")
print(f"REBUILD_E2E = {REBUILD_E2E}")
print(f"L2_SOURCE_OUT = {L2_SOURCE_OUT}")
print(f"seed={SEED}  drv={DRV_M_S:.3f} m/s  requirement={REQUIREMENT_M_S:.2f} m/s")
print(f"BULK_RV_KM_S = {BULK_RV_KM_S}")


[2026-08-19T12:36:50][hispecdrp.utils.line_mask_utils:1637][INFO]: Wrote 1785 mask lines to /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag/lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.relaxed.gradient.delta.mas
wrote permissive gradient mask: /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag/lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.relaxed.gradient.delta.mas
n_lines = 1785  (RELAXED_TEMPLATE_MASK_BUILD_KW)
REBUILD_E2E = False
L2_SOURCE_OUT = /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line
seed=0  drv=5.000 m/s  requirement=0.10 m/s
BULK_RV_KM_S = [-150.0, -100.0, -50.0, 0.0, 50.0, 100.0, 150.0]


## 3. Injection / stellar-RV helpers

If `REBUILD_E2E` is false, open the blaze L2 from `L2_SOURCE_OUT` and re-run
`StellarRVPrimitive` with the permissive mask into `OUT/<label>/`.
Shared Stage-1 WAVE is always baked at RV=0 when a full E2E rebuild is used.


In [4]:
from pathlib import Path

from hispecdrp import datamodels
import importlib

from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline
from hispecdrp.primitives import StellarRVPrimitive
from hispecdrp.simulations import e2e_pipeline as _e2e_pipeline

importlib.reload(_e2e_pipeline)
from hispecdrp.simulations.e2e_pipeline import ensure_shared_stage1_cals_for_rv_sim
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum

SHARED_CALS = None
SHARED_DIR = str(Path(OUT) / "_shared_stage1_rv0")
REBUILD_SHARED_CALS = False


def _find_blaze_l2(run_dir: Path) -> Path:
    cands = [
        p
        for p in sorted(run_dir.glob("HB.*_spec*.fits"))
        if "uncal" not in p.name and "stellar_rv" not in p.name
    ]
    if not cands:
        raise FileNotFoundError(f"No blaze L2 under {run_dir}")
    return cands[-1]


def run_stellar_rv_on_l2(
    label: str,
    injected_rv_km_s: float,
    l2_path: Path,
    *,
    save_preview: bool = False,
) -> dict:
    '''Re-run line-by-line stellar RV with MASK_PATH; write under OUT/label.'''
    run_dir = str(Path(OUT) / label)
    os.makedirs(run_dir, exist_ok=True)
    injected_m_s = float(injected_rv_km_s) * 1000.0
    print(
        f"\n=== {label}: stellar RV only  inject={injected_rv_km_s:.6f} km/s  "
        f"l2={l2_path.name} ==="
    )
    l2_in = datamodels.open(str(l2_path))
    prim = StellarRVPrimitive(
        output_dir=run_dir,
        save_result=True,
        **stellar_rv_primitives(injected_m_s, save_preview=save_preview),
        max_cores=4,
    )
    l2 = prim.apply(l2_in)
    out_spec = Path(run_dir) / Path(l2_path).name
    if out_spec.is_file():
        l2 = datamodels.open(str(out_spec))
    fib = FIBERS[0]
    avg = getattr(l2, f"rvccfavg_fib{fib}")
    rv_rec = float(np.asarray(avg["RV"]).ravel()[0])
    rv_err = float(np.asarray(avg["RV_ERR"]).ravel()[0])
    n_chunks = len(getattr(l2, f"rvccf_fib{fib}"))
    print(
        f"fiber {fib}: recovered RV = {rv_rec:.9f} +/- {rv_err:.9f} m/s  "
        f"(bias vs inject = {rv_rec - injected_m_s:.9f} m/s)  "
        f"n_chunks={n_chunks}"
    )
    return {
        "label": label,
        "run_dir": run_dir,
        "seed": SEED,
        "inject_km_s": float(injected_rv_km_s),
        "inject_m_s": injected_m_s,
        "recover_m_s": rv_rec,
        "recover_err_m_s": rv_err,
        "l2_path": str(out_spec if out_spec.is_file() else l2_path),
        "l2": l2,
        "source_l2": str(l2_path),
    }


def run_one_injection(
    label: str,
    injected_rv_km_s: float,
    seed: int,
    *,
    save_preview: bool = False,
) -> dict:
    '''Full densify/UTR/Stage0-2 path (only when REBUILD_E2E=True).'''
    global SHARED_CALS

    run_dir = str(Path(OUT) / label)
    os.makedirs(run_dir, exist_ok=True)
    injected_m_s = float(injected_rv_km_s) * 1000.0

    print(
        f"\n=== {label}: FULL E2E inject {injected_rv_km_s:.6f} km/s, "
        f"seed={seed} ==="
    )
    ramp, _cals, rate_map = create_ramp_from_spectrum(
        run_dir,
        spectrograph=SPECTROGRAPH,
        fibers=FIBERS,
        template_filename=TEMPLATE,
        mag_ab=3.05,
        filter_band="J",
        seed=seed,
        radial_velocity_km_s=injected_rv_km_s,
        barycentric_velocity_km_s=0.0,
        inject_rv_via=INJECT_RV_VIA,
        poisson_noise=POISSON_NOISE,
        convert_to_uint16=CONVERT_TO_UINT16,
        register_stage1_sims=False,
        register_wavesol=False,
    )

    if SHARED_CALS is None or REBUILD_SHARED_CALS:
        SHARED_CALS = ensure_shared_stage1_cals_for_rv_sim(
            SHARED_DIR,
            rate_map=rate_map,
            spectrograph=SPECTROGRAPH,
            fibers=FIBERS,
            known_rv_m_s=0.0,  # always bake WAVE at RV=0
            extraction_aperture=float(EXTRACTION_APERTURE),
            ramp_meta=getattr(ramp, "meta", None),
            rebuild=bool(REBUILD_SHARED_CALS),
        )

    do_fibers = [str(f) for f in FIBERS]
    l0 = Stage0Pipeline.call(ramp, output_dir=run_dir)
    l1 = Stage1Pipeline(
        output_dir=run_dir,
        primitives={
            "spec_extract": {
                "do_fibers": do_fibers,
                "method": "box",
                "extraction_aperture": int(EXTRACTION_APERTURE),
                "tracepos": SHARED_CALS["trace_pos"],
                "traceprof": SHARED_CALS["trace_prof"],
            },
            "set_wavesol": {"wavesol": SHARED_CALS["wavesol"]},
        },
    ).apply(l0)
    l2 = Stage2Pipeline(
        output_dir=run_dir,
        primitives={
            "blaze_corr": {
                "do_fibers": do_fibers,
                "fiberflat": SHARED_CALS["fiberflat"],
            },
            "sky_sub": {"run": False},
            "barycorr": {"run": False},
            "stellar_rv": stellar_rv_primitives(
                injected_m_s, save_preview=bool(save_preview)
            ),
        },
    ).apply(l1)

    fib = FIBERS[0]
    avg = getattr(l2, f"rvccfavg_fib{fib}")
    rv_rec = float(np.asarray(avg["RV"]).ravel()[0])
    rv_err = float(np.asarray(avg["RV_ERR"]).ravel()[0])
    print(
        f"fiber {fib}: recovered RV = {rv_rec:.9f} +/- {rv_err:.9f} m/s  "
        f"(bias vs inject = {rv_rec - injected_m_s:.9f} m/s)"
    )
    return {
        "label": label,
        "run_dir": run_dir,
        "seed": seed,
        "inject_km_s": float(injected_rv_km_s),
        "inject_m_s": injected_m_s,
        "recover_m_s": rv_rec,
        "recover_err_m_s": rv_err,
        "l2": l2,
        "l2_path": str(_find_blaze_l2(Path(run_dir))),
    }


def run_bulk_epoch(
    label: str, injected_rv_km_s: float, *, save_preview: bool = False
) -> dict:
    if REBUILD_E2E:
        return run_one_injection(
            label, injected_rv_km_s, SEED, save_preview=save_preview
        )
    src = _find_blaze_l2(Path(L2_SOURCE_OUT) / label)
    return run_stellar_rv_on_l2(
        label, injected_rv_km_s, src, save_preview=save_preview
    )


print("helpers ready; REBUILD_E2E =", REBUILD_E2E)


helpers ready; REBUILD_E2E = False


## 4. Run bulk grid (permissive mask)

For each bulk: epochs A=`bulk` and B=`bulk+5 m/s`.


In [5]:
import logging

for name in (
    "hispecdrp.primitives.utils.ccf",
    "hispecdrp.primitives.base_primitive",
    "hispecdrp.processing_component",
):
    logging.getLogger(name).setLevel(logging.WARNING)

RESULTS = {}
drv_km_s = DRV_M_S / 1000.0

for i_bulk, bulk_km_s in enumerate(BULK_RV_KM_S):
    bulk = float(bulk_km_s)
    lab_a = bulk_run_label(bulk, "a")
    lab_b = bulk_run_label(bulk, "b")
    save_preview = i_bulk == 0
    result_a = run_bulk_epoch(lab_a, bulk, save_preview=save_preview)
    result_b = run_bulk_epoch(lab_b, bulk + drv_km_s, save_preview=False)
    RESULTS[bulk] = {"a": result_a, "b": result_b}

print("\n=== Absolute recovery / bias ===")
for bulk in BULK_RV_KM_S:
    for which in ("a", "b"):
        r = RESULTS[float(bulk)][which]
        print(
            f"{r['label']:28s} inject = {r['inject_m_s']:12.3f} m/s  "
            f"recover = {r['recover_m_s']:.6f} +/- {r['recover_err_m_s']:.6f} m/s  "
            f"bias = {r['recover_m_s'] - r['inject_m_s']:.6f} m/s"
        )



=== bulk_m0150p0kms_a: stellar RV only  inject=-150.000000 km/s  l2=HB.20240924.00000.00_spec.fits ===
[2026-08-19T12:36:52][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:332][INFO]: Stellar RV: starting _perform algorithm=ccf_line_mask n_inputs=1 suffix=stellar_rv
[2026-08-19T12:36:52][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:372][INFO]: Stellar RV: ccf_line_mask mode — empty template arrays (line-mask CCF only)
[2026-08-19T12:36:52][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:395][INFO]: Stellar RV: opening input (path or in-memory model)
[2026-08-19T12:36:52][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:397][INFO]: Stellar RV: input open complete
[2026-08-19T12:36:52][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:398][INFO]: Stellar RV: cloning output L2 (output_model=L2Model include_data=True)
[2026-08-19T12:36:52][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:405][INFO]: Stellar RV: output L2 clone complete
[2026-08-19T12:36:52][h

## 5. Per-bulk line residual histograms + fail tracking

For each bulk, match A/B RVCCF rows, map each pair onto the nearest
rest-frame mask line, and record `residual = (RV_B − RV_A) − 5 m/s`.
A line **fails** a bulk when `|residual| ≥ 10 cm/s`.


In [6]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from hispecdrp.primitives.stellar_rv.rv_differential_utils import (
    estimate_matched_line_differential_rv,
    match_rvccf_rows_by_wavelength,
)
from hispecdrp.utils.physical_constants import C_M_S

PLOT_DIR = Path(OUT) / "diagnostics"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

n_mask = int(MASK_LAMBDA_UM.size)
resid_by_bulk = {
    float(b): np.full(n_mask, np.nan, dtype=np.float64) for b in BULK_RV_KM_S
}
measurable_by_bulk = {
    float(b): np.zeros(n_mask, dtype=bool) for b in BULK_RV_KM_S
}
fail_by_bulk = {float(b): np.zeros(n_mask, dtype=bool) for b in BULK_RV_KM_S}

SCORE_ROWS = []


def assign_mask_indices(rest_lambda_um: np.ndarray) -> np.ndarray:
    '''Nearest mask-line index within MASK_MATCH_HALF_M_S (else -1).'''
    out = np.full(rest_lambda_um.shape, -1, dtype=np.int32)
    for i, lam in enumerate(rest_lambda_um):
        if not np.isfinite(lam) or lam <= 0:
            continue
        dv = C_M_S * (MASK_LAMBDA_UM - lam) / lam
        j = int(np.nanargmin(np.abs(dv)))
        if np.isfinite(dv[j]) and abs(float(dv[j])) <= float(MASK_MATCH_HALF_M_S):
            out[i] = j
    return out


for bulk in BULK_RV_KM_S:
    bulk = float(bulk)
    pair = RESULTS[bulk]
    a, b = pair["a"], pair["b"]
    fib = FIBERS[0]
    ta = getattr(a["l2"], f"rvccf_fib{fib}")
    tb = getattr(b["l2"], f"rvccf_fib{fib}")

    diff = estimate_matched_line_differential_rv(
        ta,
        tb,
        inject_a_m_s=float(a["inject_m_s"]),
        inject_b_m_s=float(b["inject_m_s"]),
    )
    SCORE_ROWS.append(
        {
            "bulk_km_s": bulk,
            "estimated_drv": float(diff["estimated_drv_m_s"]),
            "residual_cm_s": float(diff["residual_m_s"]) * 100.0,
            "n_kept": int(diff["n_kept"]),
            "n_matched": int(diff["n_matched"]),
            "passed_agg": bool(
                np.isfinite(diff["residual_m_s"])
                and abs(float(diff["residual_m_s"])) < REQUIREMENT_M_S
            ),
        }
    )

    matched = match_rvccf_rows_by_wavelength(ta, tb)
    z_a = float(a["inject_m_s"]) / C_M_S
    rest = matched["wavelength_a_um"] / (1.0 + z_a)
    d_rv = matched["rv_b_m_s"] - matched["rv_a_m_s"]
    line_resid = d_rv - DRV_M_S
    idx = assign_mask_indices(rest)

    for k, j in enumerate(idx):
        if j < 0:
            continue
        r = float(line_resid[k])
        if not np.isfinite(r):
            continue
        prev = resid_by_bulk[bulk][j]
        if not np.isfinite(prev) or abs(r) < abs(prev):
            resid_by_bulk[bulk][j] = r
            measurable_by_bulk[bulk][j] = True
            fail_by_bulk[bulk][j] = abs(r) >= REQUIREMENT_M_S

    r_all = resid_by_bulk[bulk]
    finite = np.isfinite(r_all)
    n_meas = int(finite.sum())
    n_fail = int(np.sum(fail_by_bulk[bulk]))
    n_pass = int(np.sum(finite & ~fail_by_bulk[bulk]))

    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    vals_cm = r_all[finite] * 100.0
    ax.hist(vals_cm, bins=80, color="steelblue", edgecolor="none", alpha=0.9)
    ax.axvline(0.0, color="k", lw=1)
    ax.axvline(
        +REQUIREMENT_M_S * 100,
        color="crimson",
        ls="--",
        lw=1,
        label="±10 cm/s",
    )
    ax.axvline(-REQUIREMENT_M_S * 100, color="crimson", ls="--", lw=1)
    ax.set_xlabel(r"per-line residual $(\mathrm{RV}_B-\mathrm{RV}_A)-5$ [cm/s]")
    ax.set_ylabel("number of mask lines")
    ax.set_title(
        f"bulk={bulk:+.0f} km/s  measurable={n_meas}/{n_mask}  "
        f"pass={n_pass} fail={n_fail}  "
        f"matched-line Δ resid={SCORE_ROWS[-1]['residual_cm_s']:+.2f} cm/s"
    )
    ax.legend(loc="upper right", fontsize=9)
    fig.tight_layout()
    tag = f"{bulk:+04.0f}".replace("+", "p").replace("-", "m")
    fig_path = PLOT_DIR / f"resid_hist_bulk_{tag}kms.png"
    fig.savefig(fig_path, dpi=120)
    plt.show()
    print(f"wrote {fig_path}")

summary_full = pd.DataFrame(SCORE_ROWS)
summary_full["result"] = np.where(summary_full["passed_agg"], "PASS", "FAIL")
display(
    summary_full.assign(
        **{
            "bulk (km/s)": summary_full["bulk_km_s"].map(
                lambda b: "0" if abs(b) < 0.5 else f"{b:+.0f}"
            ),
            "residual (cm/s)": summary_full["residual_cm_s"].map(
                lambda x: f"{x:+.2f}"
            ),
            "n_chunks_kept": summary_full.apply(
                lambda s: f"{int(s.n_kept)}/{int(s.n_matched)}", axis=1
            ),
        }
    )[["bulk (km/s)", "residual (cm/s)", "n_chunks_kept", "result"]]
)


/var/folders/_s/tc8h43192s58wb4s6m2lbfcw0000gn/T/ipykernel_64667/341656957.py:118: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


wrote /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag/diagnostics/resid_hist_bulk_m150kms.png
wrote /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag/diagnostics/resid_hist_bulk_m100kms.png
wrote /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag/diagnostics/resid_hist_bulk_m050kms.png
wrote /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag/diagnostics/resid_hist_bulk_p000kms.png
wrote /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag/diagnostics/resid_hist_bulk_p050kms.png
wrote /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag/diagnostics/resid_hist_bulk_p100kms.png
wrote /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag/diagnostics/resid_hist_bulk_p150kms.png


,bulk (km/s),residual (cm/s),n_chunks_kept,result
0,-150,-35.42,531/1703,FAIL
1,-100,-25.58,538/1704,FAIL
2,-50,-28.82,553/1709,FAIL
3,0,-36.82,551/1708,FAIL
4,+50,-36.54,548/1704,FAIL
5,+100,-38.49,539/1703,FAIL
6,+150,-26.62,518/1705,FAIL


## 6. Cross-bulk fail matrix → recommended line list

- **Reject (7/7 fail):** measurable at all 7 bulks and `|resid|≥10 cm/s`
  every time.
- **Recommended keep:** measurable at all 7 bulks and fail count ≤ 2.

Writes recommended + reject masks under `OUT/`, plus a CSV of per-line
stats.


In [7]:
n_bulks = len(BULK_RV_KM_S)
fail_counts = np.zeros(n_mask, dtype=np.int32)
meas_counts = np.zeros(n_mask, dtype=np.int32)
for bulk in BULK_RV_KM_S:
    bulk = float(bulk)
    meas_counts += measurable_by_bulk[bulk].astype(np.int32)
    fail_counts += fail_by_bulk[bulk].astype(np.int32)

always_meas = meas_counts == n_bulks
reject_7of7 = always_meas & (fail_counts == n_bulks)
recommended = always_meas & (fail_counts <= MAX_FAILS_FOR_RECOMMENDED)
always_fail_when_present = (meas_counts >= 1) & (fail_counts == meas_counts)

print("=== Cross-bulk line diagnostics ===")
print(f"mask lines total              : {n_mask}")
print(f"measurable at all 7 bulks     : {int(always_meas.sum())}")
print(f"fail 7/7 (reject)             : {int(reject_7of7.sum())}")
print(
    f"recommended (meas=7, fail≤{MAX_FAILS_FOR_RECOMMENDED}): "
    f"{int(recommended.sum())}"
)
print(f"always-fail when present      : {int(always_fail_when_present.sum())}")

fig, ax = plt.subplots(figsize=(7.0, 3.8))
fc = fail_counts[always_meas]
ax.hist(
    fc,
    bins=np.arange(-0.5, n_bulks + 1.5, 1.0),
    color="darkorange",
    edgecolor="k",
)
ax.set_xlabel(f"n_bulks failed (|resid|≥{REQUIREMENT_M_S*100:.0f} cm/s)")
ax.set_ylabel("n mask lines (measurable at all 7)")
ax.set_title("Per-line fail counts across bulk grid")
fig.tight_layout()
fail_hist_path = PLOT_DIR / "fail_count_hist.png"
fig.savefig(fail_hist_path, dpi=120)
plt.show()
print(f"wrote {fail_hist_path}")

rows = []
for j in range(n_mask):
    row = {
        "mask_index": j,
        "lambda_um": float(MASK_LAMBDA_UM[j]),
        "lambda_nm": float(MASK_LAMBDA_UM[j] * 1000.0),
        "weight": float(MASK_WEIGHT[j]),
        "n_measurable": int(meas_counts[j]),
        "n_fail": int(fail_counts[j]),
        "reject_7of7": bool(reject_7of7[j]),
        "recommended": bool(recommended[j]),
    }
    for b in BULK_RV_KM_S:
        key = f"resid_cm_s_bulk_{float(b):+.0f}".replace("+", "p").replace(
            "-", "m"
        )
        val = resid_by_bulk[float(b)][j]
        row[key] = float(val * 100.0) if np.isfinite(val) else float("nan")
    rows.append(row)
line_table = pd.DataFrame(rows)
csv_path = Path(OUT) / "per_line_bulk_residuals.csv"
line_table.to_csv(csv_path, index=False)
print(f"wrote {csv_path}")


def write_delta_mas(path: Path, keep: np.ndarray, *, note: str) -> int:
    lam_nm = MASK_LAMBDA_UM[keep] * 1000.0
    w = MASK_WEIGHT[keep].astype(np.float64).copy()
    if w.size and np.nansum(w) > 0:
        w = w / np.nansum(w)
    meta = {
        "generator": "rv_recovery_sim_line_by_line_mask_diag.ipynb",
        "note": note,
        "n_lines": int(np.count_nonzero(keep)),
        "seed_mask": MASK_PATH,
        "requirement_m_s": REQUIREMENT_M_S,
        "max_fails_for_recommended": MAX_FAILS_FOR_RECOMMENDED,
        "bulk_grid_km_s": list(BULK_RV_KM_S),
        "drv_m_s": DRV_M_S,
    }
    with open(path, "w", encoding="utf-8") as f:
        f.write("# HISPEC_DRP_MASK_META " + json.dumps(meta) + "\n")
        f.write("# wavelength_vacuum_nm  weight\n")
        for x, ww in zip(lam_nm, w):
            f.write(f"{x:.10f}  {ww:.10e}\n")
    return int(np.count_nonzero(keep))


reject_path = Path(OUT) / f"{Path(TEMPLATE).stem}.reject_7of7.delta.mas"
rec_path = Path(OUT) / f"{Path(TEMPLATE).stem}.recommended.delta.mas"
n_rej = write_delta_mas(
    reject_path, reject_7of7, note="fail 7/7 at |resid|>=10 cm/s"
)
n_rec = write_delta_mas(
    rec_path,
    recommended,
    note=f"measurable all bulks & fail<={MAX_FAILS_FOR_RECOMMENDED}",
)
print(f"reject mask ({n_rej} lines): {reject_path}")
print(f"recommended mask ({n_rec} lines): {rec_path}")

display(
    line_table.loc[recommended, ["lambda_nm", "n_fail", "weight"]]
    .sort_values("lambda_nm")
    .head(20)
)
print(f"... showing 20 / {int(recommended.sum())} recommended lines")


=== Cross-bulk line diagnostics ===
mask lines total              : 1785
measurable at all 7 bulks     : 1545
fail 7/7 (reject)             : 283
recommended (meas=7, fail≤2): 98
always-fail when present      : 320
wrote /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag/diagnostics/fail_count_hist.png
wrote /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag/per_line_bulk_residuals.csv
reject mask (283 lines): /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag/lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.reject_7of7.delta.mas
recommended mask (98 lines): /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag/lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.recommended.delta.mas


/var/folders/_s/tc8h43192s58wb4s6m2lbfcw0000gn/T/ipykernel_64667/423368351.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,lambda_nm,n_fail,weight
175,989.173607,2,0.001091
182,989.553038,1,0.000228
191,990.731652,1,0.000270
224,995.020019,2,0.000684
240,997.499409,1,0.000761
261,1000.118042,2,0.000333
277,1002.848050,0,0.000545
278,1002.854825,0,0.000505
298,1005.174554,2,0.000207
299,1005.207542,2,0.000253


... showing 20 / 98 recommended lines


## 7. Re-score with recommended mask (subset of current RVCCF)

Filters each epoch's RVCCF to recommended rest-frame lines and re-runs the
matched-line differential estimator — no new CCF. This estimates how the
recommended list would score if those lines dominate.


In [8]:
REC_MASK = MaskLines.load(str(rec_path))
REC_LAMBDA_UM = np.asarray(REC_MASK.line_lambda_um, dtype=np.float64)


def filter_rvccf_to_mask(tbl, inject_m_s: float, mask_lambda_um: np.ndarray):
    '''Keep RVCCF rows whose rest lambda matches a recommended mask line.'''
    if tbl is None or len(tbl) == 0:
        return tbl
    z = float(inject_m_s) / C_M_S
    rest = np.asarray(tbl["WAVELENGTH"], dtype=np.float64) / (1.0 + z)
    keep = np.zeros(len(tbl), dtype=bool)
    for i, lam in enumerate(rest):
        if not np.isfinite(lam) or lam <= 0:
            continue
        dv = C_M_S * (mask_lambda_um - lam) / lam
        j = int(np.nanargmin(np.abs(dv)))
        if np.isfinite(dv[j]) and abs(float(dv[j])) <= float(MASK_MATCH_HALF_M_S):
            keep[i] = True
    return tbl[keep]


RESCORE = []
for bulk in BULK_RV_KM_S:
    bulk = float(bulk)
    a = RESULTS[bulk]["a"]
    b = RESULTS[bulk]["b"]
    fib = FIBERS[0]
    ta = filter_rvccf_to_mask(
        getattr(a["l2"], f"rvccf_fib{fib}"), a["inject_m_s"], REC_LAMBDA_UM
    )
    tb = filter_rvccf_to_mask(
        getattr(b["l2"], f"rvccf_fib{fib}"), b["inject_m_s"], REC_LAMBDA_UM
    )
    if len(ta) == 0 or len(tb) == 0:
        RESCORE.append(
            {
                "bulk_km_s": bulk,
                "estimated_drv": float("nan"),
                "residual_cm_s": float("nan"),
                "n_kept": 0,
                "n_matched": 0,
                "passed": False,
            }
        )
        continue
    diff = estimate_matched_line_differential_rv(
        ta,
        tb,
        inject_a_m_s=float(a["inject_m_s"]),
        inject_b_m_s=float(b["inject_m_s"]),
    )
    resid = float(diff["residual_m_s"])
    RESCORE.append(
        {
            "bulk_km_s": bulk,
            "estimated_drv": float(diff["estimated_drv_m_s"]),
            "residual_cm_s": resid * 100.0,
            "n_kept": int(diff["n_kept"]),
            "n_matched": int(diff["n_matched"]),
            "passed": bool(np.isfinite(resid) and abs(resid) < REQUIREMENT_M_S),
        }
    )

rescore_df = pd.DataFrame(RESCORE)
display(
    rescore_df.assign(
        **{
            "bulk (km/s)": rescore_df["bulk_km_s"].map(
                lambda b: "0" if abs(b) < 0.5 else f"{b:+.0f}"
            ),
            "residual (cm/s)": rescore_df["residual_cm_s"].map(
                lambda x: f"{x:+.2f}" if np.isfinite(x) else "nan"
            ),
            "n_chunks_kept": rescore_df.apply(
                lambda s: f"{int(s.n_kept)}/{int(s.n_matched)}", axis=1
            ),
            "result": np.where(rescore_df["passed"], "PASS", "FAIL"),
        }
    )[["bulk (km/s)", "residual (cm/s)", "n_chunks_kept", "result"]]
)
n_pass = int(rescore_df["passed"].sum())
print(f"Recommended-mask rescore (RVCCF subset): {n_pass}/{len(rescore_df)} PASS")
print(f"recommended mask path: {rec_path}")
print(f"per-line CSV: {csv_path}")
print(f"histograms: {PLOT_DIR}")

summary_json = {
    "mask_path": MASK_PATH,
    "n_mask_lines": n_mask,
    "n_recommended": int(recommended.sum()),
    "n_reject_7of7": int(reject_7of7.sum()),
    "requirement_m_s": REQUIREMENT_M_S,
    "full_mask_scores": SCORE_ROWS,
    "recommended_rescores": RESCORE,
    "recommended_mask": str(rec_path),
    "reject_mask": str(reject_path),
}
with open(Path(OUT) / "mask_diag_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=2)
print(f"wrote {Path(OUT) / 'mask_diag_summary.json'}")


,bulk (km/s),residual (cm/s),n_chunks_kept,result
0,-150,-6.69,28/131,PASS
1,-100,+0.18,38/128,PASS
2,-50,-12.84,36/128,FAIL
3,0,-15.41,26/130,FAIL
4,+50,-19.65,29/127,FAIL
5,+100,-21.40,30/127,FAIL
6,+150,-23.25,18/124,FAIL


Recommended-mask rescore (RVCCF subset): 2/7 PASS
recommended mask path: /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag/lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.recommended.delta.mas
per-line CSV: /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag/per_line_bulk_residuals.csv
histograms: /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag/diagnostics
wrote /Users/ckilpatrick/hispec_rv_recovery_sim/line_by_line_mask_diag/mask_diag_summary.json
